# Lab 04 — Delta Column Mapping

Schema evolution adds approved fields and widens types. Column mapping solves a different problem: it lets Delta track columns by stable metadata rather than only by their visible names. That enables metadata-only renames and safe column drops without rewriting every data file.

This notebook works on a dedicated demonstration table. It never renames or drops a column from the production Silver table. The demonstration will:

1. create a deterministic copy of clean Silver rows with name-based column mapping enabled;
2. rename `country` to `billing_country`;
3. prove row content and keys were preserved;
4. add and then safely drop a disposable `legacy_note` column;
5. validate table properties and Delta history; and
6. rerun the desired-state operations safely.

## 1. Load shared configuration

The configuration notebook supplies the catalog, schema, production Silver table, demonstration table name, and validation switch.

In [0]:
%run ./lab04_00_config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

column_mapping_table = table_names["column_mapping_demo"]
seed_view = "lab04_column_mapping_seed"
demo_row_limit = 200

print(f"Source Silver table: {silver_table}")
print(f"Column-mapping demo table: {column_mapping_table}")
print(f"Deterministic demo rows: {demo_row_limit}")

## 2. Verify the Silver prerequisite

The source must already be cleaned, typed, and deduplicated. This guard prevents the schema demonstration from hiding an upstream data-quality problem.

In [0]:
if not spark.catalog.tableExists(silver_table):
    raise FileNotFoundError(
        f"Silver table {silver_table} does not exist. "
        "Run lab04_04_silver_merge.ipynb first."
    )

silver_source_df = spark.table(silver_table)
required_columns = {
    "transaction_line_id", "invoice_no", "stock_code", "description",
    "quantity", "invoice_timestamp", "unit_price", "customer_id", "country",
}
missing_columns = sorted(required_columns - set(silver_source_df.columns))
if missing_columns:
    raise AssertionError(f"Silver prerequisite is missing columns: {missing_columns}")

silver_source_count = silver_source_df.count()
if silver_source_count == 0:
    raise ValueError(f"Silver table {silver_table} is empty.")

print(f"✅ Silver prerequisite ready: {silver_source_count:,} rows.")

## 3. Create a deterministic mapped table

A 200-row sample is selected by the stable transaction key. The table is recreated on every notebook run because it is an isolated experiment. Name-based column mapping and the compatible Delta protocol are declared when the table is created.

`legacy_note` is deliberately included so the notebook can later demonstrate a safe drop.

In [0]:
seed_df = (
    silver_source_df
    .orderBy("transaction_line_id")
    .limit(demo_row_limit)
    .select(
        "transaction_line_id", "invoice_no", "stock_code", "description",
        "quantity", "invoice_timestamp", "unit_price", "customer_id", "country",
    )
)

seed_count = seed_df.count()
if seed_count == 0:
    raise AssertionError("The deterministic column-mapping seed is empty.")

seed_df.createOrReplaceTempView(seed_view)
spark.sql(f"DROP TABLE IF EXISTS {column_mapping_table}")
spark.sql(
    f"""
    CREATE TABLE {column_mapping_table}
    USING DELTA
    TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name',
        'delta.minReaderVersion' = '2',
        'delta.minWriterVersion' = '5',
        'lab04.purpose' = 'column_mapping_demo'
    )
    AS
    SELECT *, CAST('REMOVE_AFTER_MIGRATION' AS STRING) AS legacy_note
    FROM {seed_view}
    """
)

created_count = spark.table(column_mapping_table).count()
if created_count != seed_count:
    raise AssertionError(f"Demo creation lost rows: seed={seed_count}, table={created_count}.")

print(f"✅ Name-mapped Delta table created with {created_count:,} rows.")
display(spark.table(column_mapping_table).limit(20))

## 4. Capture the pre-migration invariant

A rename must not change business data. The notebook creates a canonical record signature before the migration. After the rename, `billing_country` will be aliased back to `country` and the same signatures must remain.

In [0]:
business_columns_before = [
    "transaction_line_id", "invoice_no", "stock_code", "description",
    "quantity", "invoice_timestamp", "unit_price", "customer_id", "country",
]

def record_signatures(dataframe, ordered_columns):
    signature_expression = F.sha2(
        F.concat_ws(
            "||",
            *[F.coalesce(F.col(name).cast("string"), F.lit("<NULL>")) for name in ordered_columns],
        ),
        256,
    )
    return sorted(row["record_signature"] for row in dataframe.select(
        signature_expression.alias("record_signature")
    ).collect())

before_df = spark.table(column_mapping_table)
before_count = before_df.count()
before_distinct_keys = before_df.select("transaction_line_id").distinct().count()
before_signatures = record_signatures(before_df, business_columns_before)

if before_count != before_distinct_keys:
    raise AssertionError(
        f"Demo key uniqueness failed: rows={before_count}, distinct keys={before_distinct_keys}."
    )

print(f"Invariant captured for {before_count:,} unique rows.")

## 5. Rename `country` to `billing_country` safely

The migration is expressed as desired state. If the new name already exists, the notebook does nothing; if only the old name exists, it performs the rename. Any ambiguous state fails loudly. This makes reruns safe without silently accepting a broken schema.

In [0]:
columns_before_rename = set(spark.table(column_mapping_table).columns)

if "billing_country" in columns_before_rename and "country" not in columns_before_rename:
    rename_action = "ALREADY_RENAMED"
elif "country" in columns_before_rename and "billing_country" not in columns_before_rename:
    spark.sql(f"ALTER TABLE {column_mapping_table} RENAME COLUMN country TO billing_country")
    rename_action = "RENAMED"
else:
    raise AssertionError(
        "Unexpected rename state. Exactly one of country or billing_country must exist. "
        f"Columns={sorted(columns_before_rename)}"
    )

renamed_columns = set(spark.table(column_mapping_table).columns)
if "billing_country" not in renamed_columns or "country" in renamed_columns:
    raise AssertionError(f"Rename validation failed. Columns={sorted(renamed_columns)}")

print(f"✅ Rename desired state reached: {rename_action}.")
spark.table(column_mapping_table).printSchema()

## 6. Prove the rename preserved data

The new column is aliased to its former contract name only for comparison. Row count, unique keys, and every canonical record signature must match the pre-migration values. The old logical name must no longer be queryable.

In [0]:
after_rename_df = spark.table(column_mapping_table)
after_rename_count = after_rename_df.count()
after_rename_distinct_keys = after_rename_df.select("transaction_line_id").distinct().count()
comparison_df = after_rename_df.select(
    *[F.col(name) for name in business_columns_before if name != "country"],
    F.col("billing_country").alias("country"),
)
after_rename_signatures = record_signatures(comparison_df, business_columns_before)

if after_rename_count != before_count:
    raise AssertionError(f"Rename changed row count: before={before_count}, after={after_rename_count}.")
if after_rename_distinct_keys != before_distinct_keys:
    raise AssertionError("Rename changed business-key cardinality.")
if after_rename_signatures != before_signatures:
    raise AssertionError("Rename changed one or more business values.")

old_name_rejected = False
try:
    spark.table(column_mapping_table).select("country").limit(1).collect()
except AnalysisException:
    old_name_rejected = True

if not old_name_rejected:
    raise AssertionError("The old logical column name country is still queryable.")

print("✅ Metadata-only rename preserved every tested row and value.")
display(after_rename_df.select("transaction_line_id", "billing_country").limit(20))

## 7. Drop a disposable column safely

`legacy_note` is not part of the governed Silver contract. With column mapping enabled, `DROP COLUMN` removes it from the logical schema without rewriting all existing data files. Dropping a column is still a governed breaking change: downstream consumers must be checked before this operation is approved.

The operation below is idempotent—if the column is already absent, no additional change is made.

In [0]:
columns_before_drop = set(spark.table(column_mapping_table).columns)
if "legacy_note" in columns_before_drop:
    spark.sql(f"ALTER TABLE {column_mapping_table} DROP COLUMN legacy_note")
    drop_action = "DROPPED"
else:
    drop_action = "ALREADY_ABSENT"

after_drop_df = spark.table(column_mapping_table)
if "legacy_note" in after_drop_df.columns:
    raise AssertionError("legacy_note remains in the table schema after DROP COLUMN.")
if after_drop_df.count() != before_count:
    raise AssertionError("Dropping legacy_note changed the row count.")

post_drop_comparison_df = after_drop_df.select(
    *[F.col(name) for name in business_columns_before if name != "country"],
    F.col("billing_country").alias("country"),
)
if record_signatures(post_drop_comparison_df, business_columns_before) != before_signatures:
    raise AssertionError("Dropping legacy_note changed retained business values.")

print(f"✅ Drop desired state reached: {drop_action}; retained data is unchanged.")

## 8. Verify mapping properties and protocol

The table must explicitly report name-based mapping. The detail output also records the reader/writer protocol used by Delta to protect compatibility.

In [0]:
detail_row = spark.sql(f"DESCRIBE DETAIL {column_mapping_table}").first().asDict(recursive=True)
properties = detail_row.get("properties", {})
mapping_mode = properties.get("delta.columnMapping.mode")

if mapping_mode != "name":
    raise AssertionError(f"Expected delta.columnMapping.mode=name, found {mapping_mode!r}.")

mapping_properties_df = spark.createDataFrame(
    [
        ("delta.columnMapping.mode", mapping_mode),
        ("minReaderVersion", str(detail_row.get("minReaderVersion"))),
        ("minWriterVersion", str(detail_row.get("minWriterVersion"))),
    ],
    ["property", "value"],
)
display(mapping_properties_df)
print("✅ Name-based column mapping and compatible Delta protocol confirmed.")

## 9. Final validation and Delta history

The final summary is suitable for README evidence. Delta history should contain table creation plus schema-changing operations for the rename and drop. Exact operation labels can vary slightly by runtime, so the notebook displays the authoritative history instead of hard-coding a label.

In [0]:
final_df = spark.table(column_mapping_table)
final_columns = set(final_df.columns)
final_count = final_df.count()
final_distinct_keys = final_df.select("transaction_line_id").distinct().count()

validation_results = {
    "mapping_mode_is_name": int(mapping_mode == "name"),
    "billing_country_exists": int("billing_country" in final_columns),
    "old_country_absent": int("country" not in final_columns),
    "legacy_note_absent": int("legacy_note" not in final_columns),
    "row_count_preserved": int(final_count == before_count),
    "business_keys_unique": int(final_count == final_distinct_keys),
    "old_name_rejected": int(old_name_rejected),
}

if run_validation and any(value != 1 for value in validation_results.values()):
    raise AssertionError(f"Column-mapping validation failed: {validation_results}")

validation_df = spark.createDataFrame(
    [(name, value) for name, value in validation_results.items()],
    ["validation", "passed"],
)
display(validation_df.orderBy("validation"))
display(
    spark.sql(f"DESCRIBE HISTORY {column_mapping_table}")
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
    .orderBy(F.col("version").desc())
)
print(f"✅ Column-mapping demonstration completed with {final_count:,} preserved rows.")

## What this notebook proved

| Change | Delta command | Result | Governance requirement |
|---|---|---|---|
| Rename | `RENAME COLUMN country TO billing_country` | Logical name changed; tested rows and values preserved | Update contracts, queries, dashboards, and lineage consumers |
| Drop | `DROP COLUMN legacy_note` | Column removed from the logical schema; retained values preserved | Confirm no downstream dependency and follow retention policy |
| Replay | Desired-state checks | Repeated execution remains safe | Fail on ambiguous schema states |

> **Important:** metadata-only drop does not immediately erase the old column bytes from historical Delta files. Physical removal requires a controlled rewrite and later `VACUUM` after the retention period. Notebook 11 covers maintenance and retention safety.

### Evidence to capture

Take screenshots of:

1. the renamed schema containing `billing_country`;
2. the validation table with all checks equal to `1`; and
3. `DESCRIBE HISTORY` showing the schema changes.

## Next notebook

Continue with **`lab04_11_maintenance.ipynb`** to compare Liquid Clustering, `OPTIMIZE`, `VACUUM`, Z-Ordering, and classic partitioning, and to configure safe maintenance behavior.